# Assignment 7
**UCS420 – Cognitive Computing**

**NLP - I**

**Name:** Sumit Bansal  
**Roll No:** 1024170041


In [ ]:
import nltk
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
from collections import Counter
from nltk.util import ngrams
from nltk import pos_tag
import string

# Download necessary NLTK datasets
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')


## Dataset Generation (Starter Code)
*Note: Run your starter notebook code `generate_dataset()` and `process_ticket()` here. The following cells assume `tickets` is a list of text strings generated by the system.*


In [ ]:
# Paste your `generate_dataset()` and `process_ticket()` functions from the starter notebook here.
# tickets = generate_dataset('1024170041')

# ---------------------------------------------------------
# MOCK DATA FOR TESTING - Replace with actual tickets list
# ---------------------------------------------------------
TEST_LINES = [
    "Please email support@domain.com about the Wi-Fi issue. It isn't working on version 2.0.1.",
    "Call us at +1-800-555-0199 or visit https://help.domain.com. Confidence is 95.5%."
]
tickets = TEST_LINES.copy()


## Q.1 Preprocessing


In [ ]:
# 1. Preprocessing
# Run process_ticket() on every ticket. Create one table containing stats.
# Assuming process_ticket() does basic tokenization, we replicate the pipeline here:

def compute_stats(ticket_text):
    sentences = nltk.sent_tokenize(ticket_text)
    tokens = word_tokenize(ticket_text)
    punct_tokens = [t for t in tokens if t in string.punctuation]
    stop_words = set(stopwords.words('english'))
    final_tokens = [t.lower() for t in tokens if t not in string.punctuation and t.lower() not in stop_words]
    
    return {
        "num_sentences": len(sentences),
        "num_tokens": len(tokens),
        "num_punctuation": len(punct_tokens),
        "num_stopwords": len([t for t in tokens if t.lower() in stop_words]),
        "num_unique_tokens": len(set(tokens)),
        "num_final_tokens": len(final_tokens),
        "actual_tokens": tokens,
        "actual_punct": punct_tokens,
        "actual_stopwords": [t for t in tokens if t.lower() in stop_words],
        "actual_final": final_tokens
    }

stats_list = [compute_stats(t) for t in tickets]

# Create DataFrame for display
df_stats = pd.DataFrame(stats_list)
cols_to_sum = ['num_sentences', 'num_tokens', 'num_punctuation', 'num_stopwords', 'num_unique_tokens', 'num_final_tokens']

# Add total row
total_row = df_stats[cols_to_sum].sum()
total_row.name = 'TOTAL'
df_display = pd.concat([df_stats[cols_to_sum], total_row.to_frame().T])

print("--- Ticket Statistics Table ---")
display(df_display)

print("\n--- Selected Ticket Output ---")
selected = stats_list[0]
print("Tokens:", selected["actual_tokens"])
print("Punctuation:", selected["actual_punct"])
print("Stopwords:", selected["actual_stopwords"])
print("Final Tokens:", selected["actual_final"])


## Q.2 Regex extraction and replacement


In [ ]:
# 2. Regex Extraction and Replacement

extract_patterns = {
    "words_longer_than_5": r'\b[a-zA-Z]{6,}\b',
    "numbers_with_decimals_versions": r'\b\d+(?:\.\d+)+\b|\b\d+\b',
    "capitalized_words": r'\b[A-Z][a-zA-Z]*\b',
    "words_beginning_with_vowel": r'\b[AEIOUaeiou][a-zA-Z]*\b'
}

replace_patterns = [
    (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '<EMAIL>'),
    (r'https?://(?:[-\w.]|(?:%[\da-fA-F]{2}))+', '<URL>'),
    (r'(?:\+\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', '<PHONE>')
]

for i, line in enumerate(TEST_LINES):
    print(f"\n--- Processing Line {i+1} ---")
    print("Original Text:", line)
    
    # Extraction
    print("\nExtractions:")
    for name, pat in extract_patterns.items():
        matches = re.findall(pat, line)
        print(f"{name:35s}: {matches}")
        
    # Replacement
    subbed_text = line
    for pat, repl in replace_patterns:
        subbed_text = re.sub(pat, repl, subbed_text)
    print("\nReplaced Text:", subbed_text)


## Q.3 Custom tokenization


In [ ]:
# 3. Custom Tokenization
# Pattern covering URLs, Emails, Versions/Decimals, Phones, Hyphenated, Contractions, Standard Words
custom_pattern = r"https?://\S+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|(?:\+\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}|\b\d+(?:\.\d+)+\b|\b[A-Za-z]+(?:[-'][A-Za-z]+)+\b|\b\w+\b|\S"

tokenizer = RegexpTokenizer(custom_pattern)

def my_tokenize(text):
    return tokenizer.tokenize(text)

# Compare outputs
sample_text = "Please email support@domain.com about the Wi-Fi issue. It isn't working on version 2.0.1."
print("NLTK word_tokenize:\n", word_tokenize(sample_text))
print("\nmy_tokenize:\n", my_tokenize(sample_text))

# To run on tickets:
# all_custom_tokens = [my_tokenize(t) for t in tickets]


## Q.4 Stemming and lemmatization


In [ ]:
# 4. Stemming and Lemmatization

# Get unique final tokens (from Q1)
unique_final_tokens = list(set([t for stat in stats_list for t in stat['actual_final']]))

# Initialize stemmers and lemmatizer
port = PorterStemmer()
lanc = LancasterStemmer()

# Regexp Stemmer Suffix Choice: 'ing', 's', 'ed' are extremely common suffixes found in support tickets. 
regex_stem = RegexpStemmer('ing$|s$|ed$', min=4) 
lemmatizer = WordNetLemmatizer()

# POS tagging helper for lemmatizer
from nltk.corpus import wordnet
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

diff_list = []
for word in unique_final_tokens:
    p_stem = port.stem(word)
    l_stem = lanc.stem(word)
    r_stem = regex_stem.stem(word)
    lem = lemmatizer.lemmatize(word, get_wordnet_pos(word))
    
    # If they don't all match
    if not (p_stem == l_stem == r_stem == lem):
        diff_list.append({
            "Word": word,
            "Porter": p_stem,
            "Lancaster": l_stem,
            "Regexp": r_stem,
            "Lemmatized": lem
        })

df_stem_diff = pd.DataFrame(diff_list)
print("Words with differing stemming/lemmatization outputs:")
display(df_stem_diff)


## Q.5 Corpus statistics


In [ ]:
# 5. Corpus Statistics

# Compile sequences
S1 = [] 
for t in tickets: S1.extend(word_tokenize(t))

S2 = [w for w in S1 if w not in string.punctuation] 

stop_words = set(stopwords.words('english'))
S3 = [w.lower() for w in S2 if w.lower() not in stop_words]
S4 = [port.stem(w) for w in S3]

def calc_stats(tokens):
    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total if total > 0 else 0
    return total, unique, ttr

stats_corpus = {
    "S1 (All)": calc_stats(S1),
    "S2 (No Punct)": calc_stats(S2),
    "S3 (Final)": calc_stats(S3),
    "S4 (Porter)": calc_stats(S4)
}

df_corpus = pd.DataFrame.from_dict(stats_corpus, orient='index', columns=['Total Tokens', 'Unique Tokens', 'TTR'])
display(df_corpus)

print("\nTop 10 words S1:", [word for word, count in Counter(S1).most_common(10)])
print("Top 10 words S3:", [word for word, count in Counter(S3).most_common(10)])

# Plot log(rank) vs log(freq) for S3
freqs = [count for word, count in Counter(S3).most_common()]
ranks = range(1, len(freqs) + 1)

if len(freqs) > 0:
    plt.figure(figsize=(6,4))
    plt.plot(np.log(ranks), np.log(freqs), marker='o')
    plt.title("Zipf's Law - log(Rank) vs log(Frequency) for S3")
    plt.xlabel("log(Rank)")
    plt.ylabel("log(Frequency)")
    plt.grid(True)
    plt.show()

# Bigrams
print("Top 5 bigrams S1:", Counter(ngrams(S1, 2)).most_common(5))
print("Top 5 bigrams S3:", Counter(ngrams(S3, 2)).most_common(5))


## Q.6 Interpretation

**(a) Why does TTR change from S1 to S4 in your dataset?**  
TTR (Type-Token Ratio) naturally shifts through the stages. From S1 to S3, removing punctuation and highly frequent stopwords significantly drops the total token count while mostly retaining unique content words, which typically drives TTR upward. From S3 to S4, applying the Porter stemmer collapses morphological variants (e.g., 'working', 'works' → 'work'), which shrinks the pool of unique types and thus generally decreases the TTR again.

**(b) Give one example where `word_tokenize()` and your tokenizer differ. Explain the difference.**  
Take the phrase `"Wi-Fi"`. `word_tokenize()` splits it into three separate tokens `['Wi', '-', 'Fi']` because it treats standard punctuation like hyphens as distinct delimiters. By contrast, `my_tokenize()` uses a custom regex pattern designed to preserve hyphenated words, meaning it accurately keeps `"Wi-Fi"` completely intact as a single token.

**(c) Give one example where stemming produces an undesirable result.**  
Stemming often aggressively truncates words without considering morphology. For example, processing `"universal"` through a stemmer might yield `"univers"`, or `"organization"` could become `"organ"`. This strips away the real-world semantic meaning of the words and outputs non-dictionary fragments, rendering the resulting token much harder to interpret in downstream tasks.

**(d) Give one example where POS-aware lemmatization preserves meaning better than stemming.**  
Consider the word `"leaves"`. A rudimentary stemmer like Porter typically chops off the suffix and reduces it broadly to `"leav"`. However, a POS-aware lemmatizer checks the context: if the word is tagged as a Noun (e.g., "autumn leaves"), it reduces it perfectly to `"leaf"`; if it is tagged as a Verb (e.g., "he leaves early"), it reduces to `"leave"`. This keeps the semantic integrity intact based on its usage in the sentence.
